# AXI test 26: Gemini 3.8 Flash on ARC-AGI-3 games with the ledger's read-out in the prompt (arm C), against arm B of Test 25
**What this is.** The same four public ARC-AGI-3 games, 60 moves each, the same model, as Tests 24 and 25. Arm C is arm B (the Record, the object view, the untried list and the conductor) **plus one section in each turn: the Ledger**. The host writes down, from the moves of this run played so far, what each action did (the step and in how many of its tries), the bars that get smaller or larger in most moves, the colours that never changed and what the last move did. Nothing is taken from any other run and nothing is written by hand for a game. The structure behind the ledger is Rajnish Choubey's RFC-001 (the reading in game terms is for Sir to confirm). **Only arm C is run here**; arm B is read from the file Test 25 saved on your Drive, arms A and K from the files of Test 24.

The predictions and the reading rules were written before any call in `docs/GEMINI_ARC_C_TEST.md` (hash-locked). **This is not a measure of how well the model solves new games and not a leaderboard entry.**

**This test costs real money, and this notebook never turns billing on.** Steps: (1) type the prices in Cell 5; (2) run the pilot in Cell 6 (12 moves of one game, cents) and send me its printout; (3) only then type a budget in dollars (`BUDGET_USD`) and run Cell 7; (4) Cell 8 reads the result and makes no paid call.

Run each cell **one at a time, top to bottom**. **Keep this tab open** while a run is going. If it stops for any reason, open the notebook again, run Cells 2 to 5, then run the cell you were on again.

### Cell 1: your key (once; no code to run)
The Colab Secret `GOOGLE_API_KEY` with Notebook access on (you did this for the earlier notebooks; it is the same key). The ARC toolkit needs no key of its own: it takes an anonymous one by itself.

### Cell 2: get the code from GitHub (the update with test 26 must be pushed to GitHub first)

In [ ]:
!rm -rf /content/axi && git clone -q --depth 1 https://github.com/gaurshalendra2412/axi-runtime /content/axi
import os
assert os.path.exists("/content/axi/python/axi/experiments/gemini_arc_c.py"), "the code is not on GitHub yet: the update with test 26 must be pushed first"
print("code ok")

### Cell 3: install the Gemini library and the ARC toolkit (about 1 minute)

In [ ]:
!pip -q install -U google-genai arc-agi arcengine 2>&1 | tail -2

### Cell 4: connect Google Drive (a pop-up asks permission: Connect / Allow). Your progress and every reply are kept there

In [ ]:
import sys
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_gemini_arc")

### Cell 5: settings and the games. Fill the lines at the top. PAID is True only after you turned billing on. The budget stays empty until you have read the pilot

In [ ]:
import json, time, logging
from google.colab import userdata, files
from axi.experiments import gemini_arc as G
from axi.experiments import gemini_arc_b as B
from axi.experiments import gemini_arc_c as C
from axi.experiments.api_llm import QuotaStop, ServiceBusy, BudgetStop

PAID = None          # True ONLY after you turned billing on for this key in Google AI Studio
PRICE_IN = None      # dollars per 1,000,000 INPUT tokens of Gemini 3.8 Flash
PRICE_OUT = None     # dollars per 1,000,000 OUTPUT tokens (thinking tokens are billed as output)
PRICE_CACHED = None  # dollars per 1,000,000 CACHED input tokens (type it: without it cached tokens are counted at the full price)
BUDGET_USD = None    # the most you allow this test to cost. Leave None until you have read the pilot in Cell 6
THINKING = None      # None = the model's own default

assert PAID in (True, False), "Set PAID: True if you turned billing on, False for a free key."
try: KEY = userdata.get("GOOGLE_API_KEY")
except Exception: raise SystemExit("No key found. Do Cell 1: a Secret named GOOGLE_API_KEY with Notebook access on.")
if PAID: assert PRICE_IN and PRICE_OUT, "Type PRICE_IN and PRICE_OUT (dollars per million tokens). The notebook has no built-in price on purpose."
TAG = str(THINKING or "default"); CACHE = "axi_gemini_arc_cache_%s.jsonl" % TAG
print("reply file:", store.restore(CACHE))
guard = dict(budget_usd=BUDGET_USD, price_in=PRICE_IN, price_out=PRICE_OUT, price_cached=PRICE_CACHED) if (PAID and BUDGET_USD) else dict(price_in=PRICE_IN, price_out=PRICE_OUT, price_cached=PRICE_CACHED)
llm, MODEL = G.GL.build_chat_llm(KEY, store.path(CACHE), daily_limit=None if PAID else 20, min_interval=0.3 if PAID else 4.5, on_write=lambda: store.persist(CACHE), thinking=THINKING, **guard)
META = dict(model=MODEL, thinking=THINKING)
sp = llm.spent_usd()
print("paid:", PAID, "| model:", MODEL, "| thinking:", THINKING or "model default", "| budget:", ("$%.2f" % BUDGET_USD) if BUDGET_USD else "NOT SET (pilot only)", "| spent so far with these settings:", ("$%.2f" % sp) if sp is not None else "-")
import arc_agi
logging.disable(logging.INFO)
ARC = arc_agi.Arcade()
def make_env(game):
    env = ARC.make(game, seed=G.GAME_SEED)
    assert env is not None, "could not make " + game
    return env
for g in G.GAMES: assert any(g == str(getattr(e, "game_id", e)) for e in ARC.get_environments()), g + " is not in the list of games the toolkit offers today"
print("games ready:", ", ".join(G.GAMES))
RESET = "The free quota resets at midnight Pacific time. Then open this notebook again, run Cells 2 to 5, and run this cell again."
BUSY = "This is Google's server being busy. Wait at least 30 minutes, then run THIS SAME CELL once. Do not run it again and again."
def go(fn):
    try: return fn()
    except QuotaStop as e:
        print("\nSTOPPED FOR NOW: " + str(e))
        if isinstance(e, BudgetStop): pass
        elif isinstance(e, ServiceBusy): print(BUSY)
        else: print(RESET)
        return None
def need_budget(): assert PAID and BUDGET_USD, "Read the pilot (Cell 6), then type BUDGET_USD in Cell 5, run Cell 5 again, and run this cell again."
def save(res, name):
    p = store.path(name + ".json"); json.dump(res, open(p, "w")); store.persist(name + ".json"); return p
def persisting(name, inner):
    print("progress file:", store.restore(name + ".jsonl"))
    def cb(*a):
        inner(*a)
        if a[0] % 10 == 0 or a[0] == a[1]: store.persist(name + ".jsonl")
    return cb
def prog(t, n, r):
    if t % 5 == 0 or t == n:
        sp = llm.spent_usd(); print(f"move {t}/{n} (game {r['game']} move {r['m']}) | prompt {r['prompt_tokens']:,} (cached {r['cache_tokens']:,}) | {r['seconds']:.1f} s | {r['kind']} legal {r['reply_ok']} changed {r['changed']} | levels {r['levels']}/{r['win_levels']} | spent {('$%.2f' % sp) if sp is not None else '-'}")

### Cell 6: the PILOT. 12 real moves of arm C in the first game (cents). It prints how many moves the framework played itself, how many ledger lines the model was shown, the speed, and the projected cost of arm C at your prices

In [ ]:
assert PAID, "The pilot needs a paid key (about 12 calls)."
pc = go(lambda: C.run(llm, make_env, store.path("arc_pilot_c.jsonl"), games=(C.PILOT_GAME,), moves=C.PILOT_MOVES, meta=META))
if pc:
    store.persist("arc_pilot_c.jsonl"); C.print_stats("pilot C (ledger read-out)", pc["rows"]); C.print_games(pc["rows"])
    rep = C.pilot_report(pc["rows"], PRICE_IN, PRICE_OUT, PRICE_CACHED)
    print()
    for k, v in rep.items(): print(f"  {k}: {v if not isinstance(v, float) else round(v, 3)}")
    print("\nRead these before the budget: c_forced_share is the share of moves the framework played itself (reading rule 3 needs at most 0.4 over the whole run); c_ledger_lines_mean is how many lines of the read-out the model saw; cost_c is arm C at your prices (arm A cost $4.67 in Test 24).")
    print("Then type BUDGET_USD in Cell 5 and send me this printout before you run Cell 7.")

### Cell 7: ARM C, with the ledger read-out. 4 games x 60 moves. A line every 5 moves

In [ ]:
need_budget()
nameC = "arc_C_%s" % TAG
resC = go(lambda: C.run(llm, make_env, store.path(nameC + ".jsonl"), meta=META, max_tokens=C.C_TOKEN_CAP, progress=persisting(nameC, prog)))
if resC:
    C.print_stats("ARM C (ledger read-out)", resC["rows"]); C.print_games(resC["rows"]); store.persist(nameC + ".jsonl")
    print("spent so far on this model: $%.2f" % llm.spent_usd()); files.download(save(resC, nameC))

### Cell 8: the READING. Reads arm B from the file of Test 25 and arms A and K from the files of Test 24 (no call to the model), replays every saved move of all arms into fresh copies of the games (arm C also rebuilds the ledger), prints the tables and the 16 lines with yes / NO, and saves the report. Send me the output and the downloaded file

In [ ]:
import os
nameA, nameB, nameK = "arc_A_%s" % TAG, "arc_B_%s" % TAG, "arc_K_%s" % TAG
for n in (nameA, nameB, nameK, "arc_C_%s" % TAG): print(n, store.restore(n + ".jsonl"))
resC = resC if "resC" in globals() and resC else C.load_arm(store.path("arc_C_%s.jsonl" % TAG))
resB = B.load_arm(store.path(nameB + ".jsonl"))
resA = B.load_arm(store.path(nameA + ".jsonl"))
resK = B.load_arm(store.path(nameK + ".jsonl")) if os.path.exists(store.path(nameK + ".jsonl")) else None
ok = {}
try: C.replay(resC["rows"], make_env); ok["C"] = True
except Exception as e: print("replay C:", e); ok["C"] = False
try: B.replay(resB["rows"], make_env); ok["B"] = True
except Exception as e: print("replay B:", e); ok["B"] = False
try: G.replay(resA["rows"], make_env, "A"); ok["A"] = True
except Exception as e: print("replay A:", e); ok["A"] = False
if resK:
    try: G.replay(resK["rows"], make_env, "K"); ok["K"] = True
    except Exception as e: print("replay K:", e); ok["K"] = False
checks = C.verdict(resC, resB, resA, resK, ok)
for t, r in (("C (ledger read-out)", resC), ("B (explorer, Test 25)", resB), ("A (Record + check)", resA), ("K (kept chat)", resK)):
    if r: print(); C.print_stats("ARM " + t, r["rows"]); C.print_games(r["rows"])
C.print_verdict(checks)
print("usage:", llm.usage(), "| spent: $%.2f" % llm.spent_usd())
files.download(save(dict(C=resC, B=resB, A=resA, K=resK, checks=checks, replays=ok), "gemini_arc_c_all_%s" % TAG))

### Done
Files downloaded: the arm C file and `gemini_arc_c_all_*.json`. Upload `gemini_arc_c_all_default.json` to the chat. Nothing here says anything about any company's own service: it measures this model on these games.